# Entregável 2 — RecFair workflow (relatório)

> **Projeto:** RecFair — recomendação com contrato utilidade + justiça  
> **Arquitetura vigente:** `workflow` · `prompt_version=v2`  
> **Baseline:** `baseline` · `prompt_version=v1` (reexecutado na mesma régua)  
> **Modelo:** `gemini-3.5-flash-lite` · temperatura 0 · `golden_revision=15f3ed6986de9ce9`  
> **Data:** 13/09/2026

Notebook **somente relatório**: importa o pacote `recfair/` e `eval/`. Sem lógica de grafo, scoring ou verify duplicada.

## Como reproduzir

```bash
cd recfair
make install-dev && make kernel
cp .env.example .env   # GOOGLE_API_KEY
make chat ARCH=workflow   # ou make chat (vigente)
```

Execute este notebook do início ao fim (kernel `venv-recfair`). As células F chamam `run_eval` para baseline e workflow na mesma sessão.

# A. Estrutura herdada do Entregável 1

Reaproveitamos **sem alterar** entradas T01–T30, funções de verify e schema de saída. O que evoluiu no E2:

| Peça | Caminho | Papel |
| :--- | :--- | :--- |
| Schema | `recfair.schemas.output.RecFairOutput` | Contrato estável entre arquiteturas |
| Golden-set | `data/golden/cases.json` | T01–T30 imutáveis + **T31–T38** novos |
| Verify | `eval.verify.verify_case` | Mesma régua RF-* + gaps E2 |
| Gabarito | `eval.gold.gold_for` → `scoring/engine.py` | Fonte única de verdade offline |
| Runner | `eval.runner.run_eval` | Instrumentação + manifest em `eval/runs/` |
| Dados E2 | `tb_claims`, `tb_inventory` | Manifest `data/claims_manifest.json` |

**Correção de gabarito (não de entrada):** casos com promo/estoque/preço/claims/diversidade passam a usar o engine determinístico; o baseline E1 **não foi alterado** — só a expectativa (`gold_for`) evoluiu. Por isso a coluna `diff` na seção F mostra quantos casos o baseline erraria mesmo na régua nova.

In [1]:
import json
from pathlib import Path

from IPython.display import HTML, display

from eval.fingerprint import golden_revision, load_cases
from eval.gold import gold_for
from eval.report import (
    build_arch_comparison_table,
    build_results_table,
    render_arch_comparison,
    render_comparison_report,
    render_metrics_panel,
)
from eval.runner import run_eval
from eval.verify import verify_case
from recfair.config import apply_dotenv, ensure_google_api_key, model_version
from recfair.observability.run_record import git_sha
from recfair.schemas.output import RecFairOutput

apply_dotenv()
print("chave:", ensure_google_api_key())
cases = load_cases()
print(len(cases), "casos | golden_revision =", golden_revision(cases))
print("modelo:", model_version(), "| git:", git_sha())
manifest_path = Path("data/claims_manifest.json")
if manifest_path.is_file():
    claims_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    print(f"tb_claims: {len(claims_manifest)} SKUs com source_url")

chave: GOOGLE_API_KEY
38 casos | golden_revision = 15f3ed6986de9ce9
modelo: gemini-3.5-flash-lite | git: a522222bc150


/home/andersonbr/estudos/unicamp-llm-agents/recfair/venv-recfair/lib/python3.14/site-packages/langgraph/checkpoint/base/__init__.py:17: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


## A.1 Registro da execução (RUN_INFO)

| Campo | Valor |
| :--- | :--- |
| `architecture_id` | `baseline` (2026-09-07) vs `workflow` (2026-09-13) |
| `model` / `temperature` | `gemini-3.5-flash-lite` / 0 |
| `prompt_version` | v1 (baseline) · v2 (workflow) |
| `golden_revision` | `15f3ed6986de9ce9` (38 casos) |
| `git_sha` | registrado na célula de setup |
| ADR | [0002-workflow-scoring.md](../docs/adr/0002-workflow-scoring.md) |

A comparação abaixo usa **mesmo modelo, mesmo ambiente, baseline reexecutado na mesma sessão** — única variável controlada: arquitetura.

# B. Hipótese arquitetural

Escrita **antes** da implementação (refutada ou confirmada na seção H).

### 1. Limitação observada no E1

O baseline (`graphs/baseline.py`) faz **uma** chamada LLM com *stuffing* de `tb_catalogo` + `tb_vendas` inteiros (~29k tokens/caso). O modelo tenta agregar vendas, aplicar filtros e ranquear no prompt — tarefa **não determinística** para janela 7d, ordem de desempate, preço, estoque, claims e diversidade de marca.

### 2. Mecanismo proposto

**Workflow LangGraph determinístico:** LLM **somente** em `parse_intent` (NL → `ParsedIntent`); pipeline fixo de 7 passos em `tools/scoring/engine.py` sobre SQLite; `MemorySaver` + `thread_id` para multi-turn.

Rejeitamos ReAct (LLM escolhendo cada passo de scoring) por custo e não-determinismo; rejeitamos proxy SQL puro por não interpretar paráfrases do golden-set.

### 3. O que esperávamos melhorar

| Caso | Motivo |
| :--- | :--- |
| T05, T12 | Agregação `units_7d` + desempate por `cod_sku` |
| T17, T20, T34 | Filtro `max_price_brl` determinístico |
| T23, T35 | Exclusão `units_available=0` |
| T16, T21, T36 | Match de claims em `tb_claims` |
| T24, T37 | Flags `is_launch` / `is_promo` |
| T31–T33 | Memória de `session_intent` entre turnos |

### 4. O que esperávamos piorar

- **Latência por turno:** ainda há 1× LLM (`parse_intent`) — hipótese inicial era piora; **refutada** (ver seção F: latência caiu ~58%).
- **Complexidade:** grafo LangGraph, checkpoint, trace — aceito como custo operacional.
- **T26–T30:** guardrails de PII/injection — permanecem gap (E3).

# C. Arquitetura da v2

- [x] **workflow determinístico** — LLM só em `parse_intent`; scoring fixo via `scoring/engine.py`
- [ ] agente ReAct
- [ ] arquitetura híbrida

**Justificativa a partir do problema:** recomendação com regras de negócio explícitas (pontos, filtros, desempate) exige **reprodutibilidade** e **auditabilidade** — propriedades de workflow, não de loop ReAct.

### Grafo LangGraph (`recfair/graphs/workflow/`)

```
parse_intent (LLM)
    ├─ abstain → format_abstention → END
    └─ score → synthesize → END
```

| Requisito E2 | Implementação |
| :--- | :--- |
| Estado explícito | `WorkflowState`: query, intent, session_intent, trace, métricas |
| Fluxo >1 etapa | 4 nós + pipeline de 7 passos dentro de `score` |
| Tool real | `score_recommendation()` — leitura SQLite, filtros, pontuação |
| Decisão / roteamento | `route_after_intent`: abstain vs scoring |
| Término + limite | `recursion_limit=12` no invoke; grafo acíclico (sem loop agente-tool) |

Executar no terminal: `make chat ARCH=workflow` (ou `make chat`). Comandos: `/trace`, `/reset`, `/arch`.

### Decisão técnica central: separar interpretação de execução

| Camada | Responsável | Entrada no LLM? |
| :--- | :--- | :--- |
| NL → intent | `parse_intent` (Gemini structured) | Sim — prompt curto (~425 tokens/caso) |
| Scoring | `engine.py` (Python puro) | **Não** — catálogo/vendas/claims via código |

Isso explica a queda de **~29k → ~425 tokens de entrada/caso**: o LLM deixa de ver CSVs inteiros.

In [2]:
from recfair.graphs.registry import list_architectures

print("Arquiteturas registradas:", list_architectures())

Arquiteturas registradas: ['baseline', 'workflow']


# D. Contexto e memória

### Estado preservado

`session_intent` (`ParsedIntent` parcial) persiste no checkpoint LangGraph entre turnos: categoria, marca, `require_diversity`, `claim_terms`, etc. Turno 2 ("me mostra os mesmos de novo") **não** reenvia histórico de chat ao LLM — só merge determinístico intent atual + sessão.

### Evidência: falha **sem** memória

A célula abaixo executa T33 com **`thread_id` novo a cada turno**. Turno 2 ("me mostra os mesmos de novo") não recupera marca/categoria do turno 1 e retorna **lista vazia** — evidência de falha sem checkpoint.

### Evidência: sucesso **com** memória

Demo T33 com `thread_id` fixo — turno 2 repete exatamente o Top 5 do turno 1 (célula seguinte).

### Custo do contexto

| Arquitetura | Estratégia | Crescimento por turno |
| :--- | :--- | :--- |
| E1 baseline | Reenvia catálogo+vendas inteiros | ~29k tokens **fixos** por turno |
| E2 workflow | Só `parse_intent` + checkpoint compacto | ~425 tokens/turno; sessão em state, não no prompt |

**Contenção E2:** estado estruturado (`session_intent`) em vez de histórico textual crescente.

### Isolamento

Cada `thread_id` → bucket no `MemorySaver`. CLI gera UUID por sessão; eval usa `thread_id=case_id`; `/reset` apaga checkpoint.

In [3]:
# Demo memória: T33 — requer GOOGLE_API_KEY e arch=workflow
from recfair.graphs import workflow as workflow_mod

case_t33 = next(c for c in cases if c["id"] == "T33")
tid = "demo-t33"
workflow_mod.reset_checkpoint(tid)
for turn in case_t33["turns"]:
    out, metrics = workflow_mod.run(turn, thread_id=tid)
    skus = [i.sku for i in out.items] if out.items else []
    print(turn[:50], "...", "->", skus)

Quais os produtos de cabelo da Match mais vendidos ... -> ['F3P9W2', 'L6K1C8', '2Y8N4T', 'R5B7Q3', '9C4M1H']
Me mostra os mesmos de novo. ... -> ['F3P9W2', 'L6K1C8', '2Y8N4T', 'R5B7Q3', '9C4M1H']


In [4]:
# Demo memória: T33 SEM checkpoint — cada turno isolado
from recfair.graphs import workflow as workflow_mod

case_t33 = next(c for c in cases if c["id"] == "T33")
print("=== Sem memória (thread_id novo por turno) ===")
for i, turn in enumerate(case_t33["turns"]):
    tid = f"demo-t33-isolated-{i}"
    workflow_mod.reset_checkpoint(tid)
    out, _ = workflow_mod.run(turn, thread_id=tid)
    skus = [item.sku for item in out.items] if out.items else []
    print(f"turno {i+1}:", turn[:48], "... ->", skus)


=== Sem memória (thread_id novo por turno) ===
turno 1: Quais os produtos de cabelo da Match mais vendid ... -> ['F3P9W2', 'L6K1C8', '2Y8N4T', 'R5B7Q3', '9C4M1H']
turno 2: Me mostra os mesmos de novo. ... -> []


# E. Ferramentas e integração externa

### Decisão: tools locais, não MCP

| Critério | Tool local SQLite | MCP |
| :--- | :--- | :--- |
| Latência | In-process, sem rede | Round-trip + serialização |
| Privilégio | Leitura de CSV/SQLite empacotado | Superfície de ataque + config extra |
| Reuso multi-agente | Baixo no E2 (1 consumidor) | Justificável se N agentes compartilharem |
| Dados | Já versionados no pacote | Overhead sem ganho medido |

**Conclusão:** MCP seria prematuro — dados sintéticos/curados no repositório, consumidor único (`workflow`). MCP entra no roadmap se integração **externa ao vivo** (PDP, estoque real) for compartilhada por múltiplos agentes.

Retorno de tool = **entrada não confiável** se vier de fonte externa; no E2 as tabelas são curadas offline (manifest em `claims_manifest.json`).

### Pipeline de scoring (7 passos = 7 tools lógicas)

| Passo | Ação | Observabilidade (`action`) |
| :--- | :--- | :--- |
| 1 `filter_by_category_brand` | Inclui SKU no pool | `pool` |
| 2 `exclude_stock_and_price` | **Remove** sem estoque ou acima do teto | `removed` / `pool` |
| 3 `score_claims` | +2 se termo ∈ texto do claim | `bonus` |
| 4 `score_brand_diversity` | +1 representante por marca | `bonus` |
| 5 `add_promo_launch` | +1 launch/promo | `bonus` |
| 6 `rank_by_sales_tiebreak` | Ordena pontos → units_7d → sku | `ranked` |
| 7 `assemble_top5` | Seleciona Top N | `pool` |

Cada passo incrementa `tool_calls` no manifest; média **238 calls / 41 invocações LLM ≈ 5,8 tools/turno** — trabalho real, não stub.

In [5]:
contrato_integracao = {
    "capacidade": "Consulta determinística a catálogo, vendas 7d, claims PDP e estoque",
    "implementacao": "tool local (Python/SQLite in-process)",
    "tools": [
        {
            "nome": "score_recommendation",
            "args": {"intent": "ParsedIntent"},
            "retorno": "ScoringResult(skus, trace, points, tool_calls)",
            "erros": ["pool vazio → abstention downstream"],
        }
    ],
    "resources": ["tb_catalogo", "tb_vendas", "tb_claims", "tb_inventory"],
    "clientes_previstos": ["workflow graph (E2)", "eval/gold.py (gabarito)"],
    "justificativa": "Dados empacotados, latência mínima, sem rede; MCP só se multi-agente + fonte externa",
    "alternativa_descartada": "MCP — overhead e superfície de ataque sem reuso medido no E2",
    "privilegio": "somente leitura; escopo limitado ao diretório data/",
}

print(json.dumps(contrato_integracao, ensure_ascii=False, indent=2))


{
  "capacidade": "Consulta determinística a catálogo, vendas 7d, claims PDP e estoque",
  "implementacao": "tool local (Python/SQLite in-process)",
  "tools": [
    {
      "nome": "score_recommendation",
      "args": {
        "intent": "ParsedIntent"
      },
      "retorno": "ScoringResult(skus, trace, points, tool_calls)",
      "erros": [
        "pool vazio → abstention downstream"
      ]
    }
  ],
  "resources": [
    "tb_catalogo",
    "tb_vendas",
    "tb_claims",
    "tb_inventory"
  ],
  "clientes_previstos": [
    "workflow graph (E2)",
    "eval/gold.py (gabarito)"
  ],
  "justificativa": "Dados empacotados, latência mínima, sem rede; MCP só se multi-agente + fonte externa",
  "alternativa_descartada": "MCP — overhead e superfície de ataque sem reuso medido no E2",
  "privilegio": "somente leitura; escopo limitado ao diretório data/"
}


# F. Comparação baseline × workflow (mesma régua)

Gabarito unificado via `scoring/engine.py`. Baseline E1 **não foi alterado**; só a expectativa (`gold_for`) evoluiu para refletir regras de negócio explícitas.

**Setup experimental:** `run_eval(arch="baseline")` e `run_eval(arch="workflow")` na mesma sessão, mesmo `golden_revision`, mesmo modelo. Runs persistidos em `eval/runs/`.

### Correções de gabarito (T01–T30)

Casos com promo/estoque/preço/claims/diversidade passam a usar o engine; diferenças vs saída E1 aparecem na coluna `diff` da tabela workflow — muitos "erros" do baseline são falhas **reais** expostas pela régua nova, não regressão do workflow.

In [4]:
manifest_baseline = run_eval(arch="baseline", persist=True)

Unexpected argument 'thinking_level' provided to ChatGoogleGenerativeAI. Did you mean: 'thinking_budget'?
/home/andersonbr/estudos/unicamp-llm-agents/recfair/recfair/graphs/baseline.py:139: UserWarning: WARNING! thinking_level is not default parameter.
                thinking_level was transferred to model_kwargs.
                Please confirm that thinking_level is what you intended.
  structured = _get_structured_llm()


In [5]:
manifest_workflow = run_eval(arch="workflow", persist=True)

In [6]:
assert manifest_baseline["golden_revision"] == manifest_workflow["golden_revision"]
print("golden_revision:", manifest_baseline["golden_revision"])

golden_revision: 15f3ed6986de9ce9


In [7]:
_E1_LABELS = {
    "arch_title": "Comparação baseline × workflow (E1)",
    "arch_subtitle": "golden_revision compartilhado",
    "metrics_subtitle": "Run baseline E1",
    "rate_restrict_hint": "Escopo restrito Baseline E1",
    "rate_overall_hint": "Escopo completo Workflow E2",
    "results_title": "Baseline × gabarito (golden-set)",
}
_E2_LABELS = {
    "arch_title": "Comparação baseline × workflow (E2)",
    "arch_subtitle": "golden_revision compartilhado",
    "metrics_subtitle": "Run workflow E2",
    "rate_restrict_hint": "Escopo restrito Baseline E1",
    "rate_overall_hint": "Escopo completo Workflow E2",
    "results_title": "Workflow × gabarito (golden-set)",
}

In [8]:
display(
    HTML(
        render_arch_comparison(
            manifest_baseline,
            manifest_workflow,
            title=_E2_LABELS["arch_title"],
            subtitle=_E2_LABELS["arch_subtitle"],
            left_label="baseline",
            right_label="workflow",
            metric_prefix="e2",
        )
    )
)

métrica,baseline,workflow,delta
Restrito (S_*),0.2609,0.8261,0.5652
Scoring (T34–T38),0.2,0.8,0.6
Memória (T31–T33),0.0,0.6667,0.6667
Geral,0.1842,0.7632,0.579
Latência mediana (s),1.91,0.8,-1.11
Tokens entrada média,29423.6,425.4,-28998.2
Tokens saída média,442.0,55.8,-386.2
Chamadas LLM,41.0,41.0,0.0
Chamadas tools,0.0,238.0,238.0
Custo est. (USD),0.377418,0.009879,-0.37


In [9]:
display(
    HTML(
        render_metrics_panel(
            manifest_baseline["resumo"],
            metric_prefix="e1",
            subtitle=_E2_LABELS["metrics_subtitle"],
            rate_restrict_hint=_E2_LABELS["rate_restrict_hint"],
            rate_overall_hint=_E2_LABELS["rate_overall_hint"],
        )
    )
)

display(
    HTML(
        render_metrics_panel(
            manifest_workflow["resumo"],
            metric_prefix="e2",
            subtitle=_E2_LABELS["metrics_subtitle"],
            rate_restrict_hint=_E2_LABELS["rate_restrict_hint"],
            rate_overall_hint=_E2_LABELS["rate_overall_hint"],
        )
    )
)

e1_rate_restrictEscopo restrito Baseline E1,6/23 (26.1%)
e1_rate_overallEscopo completo Workflow E2,7/38 (18.4%)
Latência mediana,1.91 s
Latência média,2.01 s
Chamadas LLM,41
Tokens entrada / saída,"1,118,095 / 16,796"
Custo estimado (USD),$0.3774


e2_rate_restrictEscopo restrito Baseline E1,19/23 (82.6%)
e2_rate_overallEscopo completo Workflow E2,29/38 (76.3%)
Latência mediana,0.8 s
Latência média,0.83 s
Chamadas LLM,41
Tokens entrada / saída,"15,738 / 2,063"
Custo estimado (USD),$0.0099


In [10]:
tabela_wf = build_results_table(
    manifest_workflow["records"],
    output_column="workflow",
    experiment="e2",
)
display(
    HTML(
        render_comparison_report(
            tabela_wf,
            "status_final",
            title=_E2_LABELS["results_title"],
            code_columns=frozenset({"workflow", "gabarito"}),
        )
    )
)

caso,tipo caso,tipo teste,status final,workflow,gabarito,diff,motivo erro
T01,normal,restrito,sucesso,F3P9W2 → A8T3K5 → 3G7P2W → H8Q3N1 → L6K1C8,F3P9W2 → A8T3K5 → 3G7P2W → H8Q3N1 → L6K1C8,igual ao gabarito,—
T02,paráfrase,restrito,sucesso,F3P9W2 → A8T3K5 → 3G7P2W → H8Q3N1 → L6K1C8,F3P9W2 → A8T3K5 → 3G7P2W → H8Q3N1 → L6K1C8,igual ao gabarito,—
T03,composto,restrito,sucesso,F3P9W2 → L6K1C8 → 2Y8N4T → R5B7Q3 → 9C4M1H,F3P9W2 → L6K1C8 → 2Y8N4T → R5B7Q3 → 9C4M1H,igual ao gabarito,—
T04,normal,restrito,sucesso,24A51X → Z5C1R8 → X1Q8V3 → 7H5A2E → K8M2Q1,24A51X → Z5C1R8 → X1Q8V3 → 7H5A2E → K8M2Q1,igual ao gabarito,—
T05,janela_7d,restrito,sucesso,7K2N9A → 2M7K4F → 8V4C6N → H3L9Q1 → P1T8R5,7K2N9A → 2M7K4F → 8V4C6N → H3L9Q1 → P1T8R5,igual ao gabarito,—
T06,informação ausente,restrito,sucesso,abstention · missing_category,abstention · missing_category,—,—
T07,informação ausente,restrito,sucesso,abstention · unknown_brand,abstention · unknown_brand,—,—
T08,ambíguo,restrito,sucesso,abstention · missing_category,abstention · missing_category,—,—
T09,ambíguo,restrito,erro,7K2N9A → Q4H8L2 → 3R1B6M → 5J8P2X → W9C5TD,abstention · missing_category,—,"RF-04/05: deveria abster, obteve status=recommendation"
T10,fora de escopo,restrito,sucesso,abstention · unknown_category,abstention · unknown_category,—,—


## F.1 Interpretação dos resultados

### Ganho de performance (qualidade)

Com **n=38**, evitar generalizações estatísticas; reportamos **casos concretos**:

| Eixo | Baseline | Workflow | Δ | Evidência |
| :--- | ---: | ---: | ---: | :--- |
| Restrito S_* (T01–T15, T31–T38) | 6/23 (26,1%) | 19/23 (82,6%) | **+56,5 pp** | T05/T12 janela 7d; T34–T37 scoring |
| Memória T31–T33 | 0/3 | 2/3 | **+66,7 pp** | T32/T33 OK; T31 falha merge session |
| Scoring T34–T38 | 1/5 | 4/5 | **+60 pp** | Preço/estoque/claims/promo determinísticos |
| Geral | 7/38 (18,4%) | 29/38 (76,3%) | **+57,9 pp** | +22 casos na mesma régua |

**Casos que o baseline errava e o workflow acerta (amostra):** T05 (janela 7d), T12 (composto), T16/T23/T24/T25 (dimensões globais), T32/T33 (memória), T34–T37 (scoring isolado).

**Casos ainda falhos no workflow:** T09 (ambiguidade → deveria abster), T14 (RF-06 fairness), T17/T20 (`max_price_brl` não extraído pelo LLM), T18/T21/T22 (claims — regex frágil), T31 (memória parcial), T38 (critério família diversidade).

### Redução de custo e latência

| Métrica | Baseline | Workflow | Δ | Mecanismo |
| :--- | ---: | ---: | ---: | :--- |
| Latência mediana | 1,91 s | 0,80 s | **−58%** | Prompt pequeno + scoring local |
| Tokens entrada (média/caso) | 29 424 | 425 | **−98,6%** | Fim do stuffing de CSV no prompt |
| Tokens saída (média/caso) | 442 | 56 | **−87%** | Structured output compacto |
| Chamadas LLM | 41 | 41 | 0 | 1× `parse_intent` por turno |
| Chamadas tools | 0 | 238 | +238 | Pipeline 7 passos em código |
| Custo estimado (run completo) | $0,377 | $0,010 | **−97%** | Dominado por tokens de entrada no E1 |

O ganho de **qualidade e custo** vem de **tirar a agregação do LLM**, não de reduzir chamadas LLM. Tools substituem raciocínio probabilístico por código auditável.

### Observabilidade

`ScoreTrace` registra por SKU/passo: inclusão (`pool`), exclusão (`removed`), bônus (`bonus`), ranking (`ranked`). Ver seção G.

### Workflow determinístico = controle

Ordem fixa dos 7 passos; desempate `(pontos ↓, units_7d ↓, cod_sku ↑)` em Python — **mesmo intent → mesma saída**.

### Limitações da medida

- Conjunto pequeno; 1 acerto ≈ 2,6 pp no geral.
- Gabarito E2 recalculado — baseline penalizado na régua nova (comparação justa entre arquiteturas).
- T26–T30 passam sem guardrail real — gabarito assume sanitização futura.

# G. Novos modos de falha

Autonomia do workflow introduz falhas **distintas** do baseline monolítico:

| Modo de falha | Ocorreu? | Caso / descrição |
| :--- | :---: | :--- |
| Ferramenta correta, argumento errado | **Sim** | T17/T20: pipeline OK, mas `max_price_brl=None` — `parse_intent` não extraiu preço |
| Ferramenta chamada sem necessidade | Não | 7 passos fixos — custo CPU local, não LLM |
| Ferramenta necessária não chamada | Não | Sem seleção ReAct |
| Erro dentro da tool silencioso | Não | Pool vazio → trace + abstention |
| Laço interrompido por `recursion_limit` | Não | Grafo acíclico; limite 12 nunca atingido |
| Resposta ignora output da tool | Não | `synthesize` só usa `ScoringResult` |
| **parse_intent** errado | **Sim** | T09: recomenda sem categoria clara |
| Match claim regex frágil | **Sim** | T18/T21/T22: substring não captura paráfrases |
| Perda parcial de memória | **Sim** | T31: turno 2 perde parte do `session_intent` |
| Guardrail ausente | **Sim** | T26–T30: gap E3 (PII/injection/jailbreak) |

### Trace de scoring (T36 — claim anticaspa)

In [11]:
from eval.case_intent import intent_from_case
from recfair.tools.scoring.engine import score_recommendation

case_t36 = next(c for c in cases if c["id"] == "T36")
result = score_recommendation(intent_from_case(case_t36))
for entry in result.trace.to_dicts():
    if entry.get("sku") in {"H8Q3N1", "-"} or entry.get("action") == "bonus":
        print(entry)

{'step': 'filter_by_category_brand', 'sku': 'H8Q3N1', 'action': 'pool', 'reason': 'category=cabelos', 'points_delta': 0, 'points_total': 0}
{'step': 'exclude_stock_and_price', 'sku': 'H8Q3N1', 'action': 'pool', 'reason': 'in stock', 'points_delta': 0, 'points_total': 0}
{'step': 'score_claims', 'sku': 'H8Q3N1', 'action': 'bonus', 'reason': "match claim 'anticaspa'", 'points_delta': 2, 'points_total': 2}
{'step': 'score_brand_diversity', 'sku': 'F3P9W2', 'action': 'bonus', 'reason': 'brand representative (Match)', 'points_delta': 1, 'points_total': 1}
{'step': 'score_brand_diversity', 'sku': 'H8Q3N1', 'action': 'bonus', 'reason': 'brand representative (Malbec)', 'points_delta': 1, 'points_total': 3}
{'step': 'score_brand_diversity', 'sku': 'A8T3K5', 'action': 'bonus', 'reason': 'brand representative (Cuide-se Bem)', 'points_delta': 1, 'points_total': 1}
{'step': 'add_promo_launch', 'sku': '3G7P2W', 'action': 'bonus', 'reason': 'is_promo=true', 'points_delta': 1, 'points_total': 1}
{'ste

## G.1 Observabilidade — cada inclusão e exclusão

O trace abaixo mostra **por que** `H8Q3N1` lidera: +2 claims + representante Malbec. Produtos **removidos** no passo 2 (`removed`) explicam filtros de estoque/preço — essencial para auditar T17/T35 sem reexecutar manualmente.

Trace append-only → manifest JSON → CLI `/trace`.

# H. Análise arquitetural e pergunta obrigatória

### 1. A hipótese da seção B confirmou-se?

**Sim, parcialmente — com surpresa positiva em custo/latência.**

| Expectativa | Resultado | Evidência |
| :--- | :--- | :--- |
| Melhorar agregação/filtros | **Confirmado** | +56,5 pp restrito; T05/T12/T34–T37 |
| Melhorar memória multi-turn | **Parcial** | 2/3 (T32/T33 OK, T31 falha merge) |
| Melhorar claims | **Parcial** | T16/T36 OK; T18/T21/T22 falham (regex + intent) |
| Piorar latência | **Refutado** | 1,91 s → 0,80 s mediana |
| Guardrails T26–T30 | **Gap permanece** | Passam sem `sanitize_pii` |

### 2. Limitações resolvidas, persistentes e novas

**Resolvidas:** agregação 7d determinística; filtros inventory; memória de intent; auditabilidade (`ScoreTrace`).

**Persistentes:** NL em `parse_intent`; match claims por substring; fairness RF-06 (T14); guardrails (T26–T30).

**Novas:** acoplamento intent→scoring; complexidade LangGraph; régua engine vs saída E1.

### 3. Acoplamentos

- `ParsedIntent` compartilhado entre LLM, checkpoint e engine.
- `eval/gold.py` e runtime usam o mesmo `score_recommendation`.
- Claims offline acoplados a `_claim_match` literal.

### 4. O ganho compensou custo e complexidade?

**Sim.** +22 acertos, −97% custo, −58% latência, observabilidade total — complexidade LangGraph < custo de ~29k tokens/caso no E1.

---

### Pergunta obrigatória (Entregável 3)

> **Que responsabilidade do sistema atual seria a melhor candidata a se tornar um agente especializado na próxima versão, e por quê?**

**Candidata principal: agente de match de claims.**

Hoje `_claim_match` usa **substring literal** (`term in blob`) — frágil: não identifica sinônimos, typos ou nuances ("controle de caspa" vs "anticaspa"). Falhas T18/T21/T22 confirmam. Um agente especializado em **similaridade semântica** entre `claim_terms` e fichas `tb_claims` desacopla interpretação lexical da pipeline de pontos, mantendo o workflow determinístico downstream.

**Candidata complementar: agente de segurança e privacidade** com guardrails **antes** de `parse_intent`:
- Sanitização PII (T26–T27).
- Detecção prompt injection / jailbreak (T28–T30).
- Fail-closed com auditoria separada do agente recomendador.

E3 natural: supervisor `{security_agent → intent_agent → scoring_workflow}`.

---

## Próximos passos (multi-agente)

| Agente | Problema | Benefício |
| :--- | :--- | :--- |
| **Claim matcher** | Regex frágil | Paráfrases, linguagem coloquial, termos próximos |
| **Security / privacy** | Sem guardrails sólidos | Anti-injection, anti-jailbreak, PII fora do LLM |
| Fairness auditor *(futuro)* | RF-06 manual | Validação pós-ranking independente |

---

### Checklist de entrega E2

- [x] Estrutura herdada (A)
- [x] Hipótese (B)
- [x] Workflow LangGraph (C)
- [x] Memória com/sem checkpoint (D)
- [x] Tools locais + contrato (E)
- [x] Comparação reexecutada (F)
- [x] Modos de falha (G)
- [x] Análise final (H)